In [1]:
import torch
import torch.nn as nn
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Fijar semilla general
torch.manual_seed(42)

# ==========================================
# 1. Preparar datos
# ==========================================
X, y = load_iris(return_X_y=True)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

esc = StandardScaler()
X_train = esc.fit_transform(X_train)
X_test = esc.transform(X_test)

X_train = torch.tensor(X_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.long)
y_test = torch.tensor(y_test, dtype=torch.long)


# ==========================================
# 2. Definición de ambas redes
# ==========================================
# Red original: 1 capa oculta (16 neuronas)
class RedOriginal(nn.Module):
    def __init__(self):
        super().__init__()
        self.capas = nn.Sequential(
            nn.Linear(4, 16),
            nn.ReLU(),
            nn.Linear(16, 3),
        )

    def forward(self, x):
        return self.capas(x)


# Red modificada: 2 capas ocultas (16 y 8 neuronas)
class RedModificada(nn.Module):
    def __init__(self):
        super().__init__()
        self.capas = nn.Sequential(
            nn.Linear(4, 16),
            nn.ReLU(),
            nn.Linear(16, 8),  # Segunda capa oculta añadida
            nn.ReLU(),         # Activación ReLU añadida
            nn.Linear(8, 3),   # Capa de salida conectada a las 8 neuronas
        )

    def forward(self, x):
        return self.capas(x)


# ==========================================
# 3. Función de entrenamiento y evaluación
# ==========================================
def entrenar_y_evaluar(modelo, nombre):
    torch.manual_seed(42)
    criterio = nn.CrossEntropyLoss()
    optimizador = torch.optim.Adam(modelo.parameters(), lr=0.01)

    for epoca in range(1, 201):
        modelo.train()
        salida = modelo(X_train)
        perdida = criterio(salida, y_train)

        optimizador.zero_grad()
        perdida.backward()
        optimizador.step()

    modelo.eval()
    with torch.no_grad():
        preds = modelo(X_test).argmax(dim=1)
        acc = (preds == y_test).float().mean().item()

    return perdida.item(), acc


# ==========================================
# 4. Comparación
# ==========================================
loss_orig, acc_orig = entrenar_y_evaluar(RedOriginal(), "Original")
loss_mod, acc_mod = entrenar_y_evaluar(RedModificada(), "Modificada (2 capas)")

print("=== COMPARACIÓN DE ARQUITECTURAS ===")
print(f"{'Modelo':<30} | {'Pérdida final (Loss)':<22} | {'Exactitud (Accuracy)':<20}")
print("-" * 78)
print(f"{'Original (4 -> 16 -> 3)':<30} | {loss_orig:<22.4f} | {acc_orig:<20.3f} ({acc_orig*100:.1f}%)")
print(f"{'Modificada (4 -> 16 -> 8 -> 3)':<30} | {loss_mod:<22.4f} | {acc_mod:<20.3f} ({acc_mod*100:.1f}%)")

=== COMPARACIÓN DE ARQUITECTURAS ===
Modelo                         | Pérdida final (Loss)   | Exactitud (Accuracy)
------------------------------------------------------------------------------
Original (4 -> 16 -> 3)        | 0.0432                 | 1.000                (100.0%)
Modificada (4 -> 16 -> 8 -> 3) | 0.0292                 | 0.967                (96.7%)
